**실행 설정**

입력은 `config/paths.local.json` 또는 `SLC7A5_DATA_ROOT`로 지정합니다. 결과는 이 저장소의 `results/`에 기록합니다. 입력/실행 설명은 저장소 루트 README를 참고하세요. 이 노트북에는 기존 탐색/대체 Figure 셀이 포함되어 있으며 전체 Figure 재현 검증은 별도 단계입니다.

In [ ]:
# Repository paths; inputs are read-only, new files stay in results/.
from pathlib import Path
import os
import sys
_start = Path(os.environ.get("SLC7A5_REPO_ROOT", str(Path.cwd()))).resolve()
_repo = next((p for p in [_start, *_start.parents] if (p / "slc7a5_paths.py").is_file()), None)
if _repo is None:
    raise RuntimeError("Open this notebook inside the repository or set SLC7A5_REPO_ROOT.")
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo))
from slc7a5_paths import artifact_path, dataset_path, input_path, output_path, output_dir, setup_notebook
setup_notebook('pbmc')


# PBMC reclustering and re-annotation

`PBMC_v2_annotation.h5ad`는 이미 preprocessing/annotation이 들어있는 객체이므로, 기본 workflow는 기존 `X_scVI` latent representation을 사용해 clustering과 UMAP을 다시 계산합니다.

필요하면 `REPROCESS_FROM_COUNTS = True`로 바꿔서 `layers["counts"]`에서 normalization, HVG, PCA까지 다시 수행할 수 있습니다.


In [ ]:
# -----------------------------------------------------------------------------
# Imports and global settings
# -----------------------------------------------------------------------------
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt
from IPython.display import display

warnings.filterwarnings("ignore")
sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=120, facecolor="white", frameon=False)

SEED = 42
np.random.seed(SEED)


In [ ]:
# -----------------------------------------------------------------------------
# Parameters
# -----------------------------------------------------------------------------
PROJECT_DIR = Path(output_dir('02_SLC7A5_mono_Journal'))
DATA_PATH = Path(dataset_path("pbmc_annotated"))
OUT_DIR = PROJECT_DIR / "02_Analysis" / "pbmc" / "reclustered"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_NAME = "PBMC_v2"

# Default: use existing preprocessing/scVI latent and redo clustering/annotation only.
REPROCESS_FROM_COUNTS = False
CLUSTER_USE_REP = "X_scVI"

# Used when REPROCESS_FROM_COUNTS=True, or when CLUSTER_USE_REP is unavailable.
MIN_GENES_PER_CELL = 250
MIN_COUNTS_PER_CELL = 500
MAX_PCT_MT = 20
MIN_CELLS_PER_GENE = 3
N_HVG = 3000
N_PCS = 50

N_NEIGHBORS = 15
LEIDEN_RESOLUTION = 0.8
CLUSTER_KEY = f"leiden_recluster_r{LEIDEN_RESOLUTION}"
NEIGHBORS_KEY = "neighbors_recluster"
UMAP_KEY = "X_umap_recluster"

RUN_CELLTYPIST = True
DOWNLOAD_CELLTYPIST_MODEL = True
CELLTYPIST_MODEL = "Immune_All_Low.pkl"

# Manual labels can be edited after reviewing marker plots and CellTypist crosstab.
# Example: {"0": "CD4 T", "1": "Classical monocyte"}
MANUAL_CLUSTER_LABELS = {}


In [ ]:
# -----------------------------------------------------------------------------
# Load annotated PBMC AnnData
# -----------------------------------------------------------------------------
adata = sc.read_h5ad(DATA_PATH)
print(f"Loaded: {DATA_PATH}")
print(f"Cells: {adata.n_obs:,}, genes: {adata.n_vars:,}")
print(adata)

print("\n.obs columns")
display(pd.Series(adata.obs.columns, name="obs_columns").to_frame())
print("\n.obsm keys:", list(adata.obsm.keys()))
print("layers:", list(adata.layers.keys()))


In [ ]:
# -----------------------------------------------------------------------------
# Keep a lightweight copy of the original labels for comparison
# -----------------------------------------------------------------------------
ORIGINAL_LABEL_COLS = [
    col for col in [
        "cellannot_final",
        "majority_voting",
        "predicted_labels",
        "leiden_scVI_1.0",
    ]
    if col in adata.obs.columns
]
print("Original label columns:", ORIGINAL_LABEL_COLS)

if "cellannot_final" in adata.obs.columns and "X_umap" in adata.obsm:
    sc.pl.umap(adata, color=["cellannot_final"], title="Original annotation")


## Build the object used for reclustering

기본값은 메모리 사용을 줄이기 위해 로드한 `adata` 자체에 새 clustering 결과를 추가합니다. 원본 파일은 `write_h5ad`를 호출하기 전까지 바뀌지 않습니다.


In [ ]:
# -----------------------------------------------------------------------------
# Optional full preprocessing from raw counts
# -----------------------------------------------------------------------------
def _copy_matrix(x):
    return x.copy() if hasattr(x, "copy") else x


def make_count_adata(source: ad.AnnData) -> ad.AnnData:
    """Create a fresh AnnData from counts stored in layers['counts'] or adata.raw."""
    if "counts" in source.layers:
        out = ad.AnnData(
            X=_copy_matrix(source.layers["counts"]),
            obs=source.obs.copy(),
            var=source.var.copy(),
        )
        out.uns["raw_reprocess_source"] = "layers['counts']"
    elif source.raw is not None:
        out = source.raw.to_adata().copy()
        out.obs = source.obs.copy()
        out.uns["raw_reprocess_source"] = "adata.raw"
    else:
        raise ValueError("No count source found. Need layers['counts'] or adata.raw.")

    out.obs_names_make_unique()
    out.var_names_make_unique()
    out.layers["counts"] = _copy_matrix(out.X)
    return out


def preprocess_from_counts(source: ad.AnnData) -> ad.AnnData:
    out = make_count_adata(source)

    out.var["mt"] = out.var_names.str.upper().str.startswith("MT-")
    out.var["ribo"] = out.var_names.str.upper().str.startswith(("RPS", "RPL"))
    out.var["hb"] = out.var_names.str.upper().str.match(r"^HB[^(P)]")

    sc.pp.calculate_qc_metrics(
        out,
        qc_vars=["mt", "ribo", "hb"],
        percent_top=None,
        log1p=False,
        inplace=True,
    )

    n_before = out.n_obs
    sc.pp.filter_cells(out, min_genes=MIN_GENES_PER_CELL)
    out = out[out.obs["total_counts"] >= MIN_COUNTS_PER_CELL].copy()

    max_counts = out.obs["total_counts"].quantile(0.99)
    out = out[(out.obs["pct_counts_mt"] < MAX_PCT_MT) & (out.obs["total_counts"] < max_counts)].copy()
    sc.pp.filter_genes(out, min_cells=MIN_CELLS_PER_GENE)

    print(f"Cells before filtering: {n_before:,}")
    print(f"Cells after filtering : {out.n_obs:,}")
    print(f"Genes after filtering : {out.n_vars:,}")

    sc.pp.normalize_total(out, target_sum=1e4)
    sc.pp.log1p(out)
    out.layers["log1p_norm"] = _copy_matrix(out.X)
    out.raw = out

    sc.pp.highly_variable_genes(out, n_top_genes=N_HVG, flavor="seurat", subset=False)
    sc.pp.scale(out, max_value=10)
    sc.tl.pca(out, n_comps=N_PCS, svd_solver="arpack", random_state=SEED)
    return out


if REPROCESS_FROM_COUNTS:
    adata_re = preprocess_from_counts(adata)
    use_rep = "X_pca"
else:
    adata_re = adata
    if CLUSTER_USE_REP in adata_re.obsm:
        use_rep = CLUSTER_USE_REP
    elif "X_pca" in adata_re.obsm:
        use_rep = "X_pca"
    else:
        print(f"{CLUSTER_USE_REP} and X_pca were not found. Running count-based preprocessing instead.")
        adata_re = preprocess_from_counts(adata)
        use_rep = "X_pca"

print(f"Using representation for clustering: {use_rep}")
print(adata_re)


In [ ]:
# -----------------------------------------------------------------------------
# QC overview when QC columns are available
# -----------------------------------------------------------------------------
qc_cols = [
    col for col in [
        "n_genes_by_counts",
        "total_counts",
        "pct_counts_mt",
        "pct_counts_ribo",
        "pct_counts_hb",
        "nFeature_RNA",
        "nCount_RNA",
        "percent.mito",
        "percent.ribo",
    ]
    if col in adata_re.obs.columns
]

if qc_cols:
    display(adata_re.obs[qc_cols].describe().T)
    plot_cols = [col for col in ["n_genes_by_counts", "total_counts", "pct_counts_mt"] if col in adata_re.obs.columns]
    if plot_cols:
        sc.pl.violin(adata_re, plot_cols, jitter=0.4, multi_panel=True)
else:
    print("No standard QC columns found.")


In [ ]:
# -----------------------------------------------------------------------------
# Recompute neighbors, Leiden clustering, and UMAP
# -----------------------------------------------------------------------------
if "X_umap" in adata_re.obsm and "X_umap_original" not in adata_re.obsm:
    adata_re.obsm["X_umap_original"] = adata_re.obsm["X_umap"].copy()

sc.pp.neighbors(
    adata_re,
    n_neighbors=N_NEIGHBORS,
    n_pcs=None if use_rep.startswith("X_scVI") else N_PCS,
    use_rep=use_rep,
    key_added=NEIGHBORS_KEY,
    random_state=SEED,
)

sc.tl.leiden(
    adata_re,
    resolution=LEIDEN_RESOLUTION,
    key_added=CLUSTER_KEY,
    neighbors_key=NEIGHBORS_KEY,
    random_state=SEED,
)

sc.tl.umap(adata_re, neighbors_key=NEIGHBORS_KEY, random_state=SEED)
adata_re.obsm[UMAP_KEY] = adata_re.obsm["X_umap"].copy()

if "X_umap_original" in adata_re.obsm:
    adata_re.obsm["X_umap"] = adata_re.obsm["X_umap_original"].copy()

print(adata_re.obs[CLUSTER_KEY].value_counts().sort_index())


In [ ]:
adata

In [ ]:
adata_re

In [ ]:
sc.pl.umap(adata, color=["cellannot_final"], title="Original annotation")

In [ ]:
sc.pl.umap(adata, color=["majority_voting"], title="celltyist")

In [ ]:
sc.pl.umap(adata, color=["leiden_recluster_r0.8"], title="leiden")

In [ ]:
adata.write(output_path('02_SLC7A5_mono_Journal/01_Data/pbmc_adata.h5ad'))

In [ ]:
sc.tl.rank_genes_groups(
    adata,
    groupby='leiden_recluster_r0.8',
    method='wilcoxon',
    use_raw=True,
    pts=True
)

sc.pl.rank_genes_groups_dotplot(adata, n_genes=5)
deg_pbmc = sc.get.rank_genes_groups_df(adata, group=None)
deg_pbmc

In [ ]:
top_genes = (
    deg_pbmc.groupby("group")["names"]
    .apply(lambda x: x.head(20))
    .explode()
    .unique()
    .tolist()
)

adata_plot = adata[:, top_genes].copy()
sc.pp.scale(adata_plot, zero_center=True, max_value=3)

In [ ]:
sc.pl.heatmap(
    adata_plot,
    var_names=top_genes,
    groupby="leiden_recluster_r0.8",
    use_raw=False,
    cmap="bwr",
    vmin=-3,
    vmax=3,
    swap_axes=True,
    dendrogram=False,
    show=True,
    figsize=(12, 7)
)

In [ ]:
print(adata.X.min(), adata.X.max())

In [ ]:
deg_pbmc.to_csv(output_path('02_SLC7A5_mono_Journal/02_Analysis/pbmc/pbmc_total_leiden0.8_DEG.csv'))

In [ ]:
# -----------------------------------------------------------------------------
# Visualize reclustering results
# -----------------------------------------------------------------------------
color_cols = [CLUSTER_KEY]
color_cols += [col for col in ["disease", "sample", "cellannot_final", "majority_voting"] if col in adata_re.obs.columns]

sc.pl.embedding(
    adata_re,
    basis="umap_recluster",
    color=color_cols,
    wspace=0.4,
    ncols=2,
)


In [ ]:
sc.pl.umap(adata, color=["leiden_recluster_r0.8"], title="leiden")

In [ ]:
from scipy.stats import fisher_exact
import pandas as pd

state = adata.obs['disease']
cluster = adata.obs['leiden_recluster_r0.8']

clusters = cluster.unique()
results = []

for clus in clusters:
    donor_in = ((state == "Normal") & (cluster == clus)).sum()
    donor_out = ((state == "Normal") & (cluster != clus)).sum()
    ipf_in = ((state == "MASLD") & (cluster == clus)).sum()
    ipf_out = ((state == "MASLD") & (cluster != clus)).sum()

    table = [[donor_in, donor_out], [ipf_in, ipf_out]]
    oddsratio, pval = fisher_exact(table)

    donor_frac = donor_in / (donor_in + donor_out) if (donor_in + donor_out) > 0 else 0
    ipf_frac = ipf_in / (ipf_in + ipf_out) if (ipf_in + ipf_out) > 0 else 0

    results.append({
        "cluster": clus,
        "donor_fraction": donor_frac,
        "ipf_fraction": ipf_frac,
        "odds_ratio": oddsratio,
        "p_value": pval
    })

df_result = pd.DataFrame(results)

df_result["cluster"] = pd.Categorical(
    df_result["cluster"],
    #categories=desired_order,
    ordered=True,
)
df_result = df_result.sort_values("cluster").reset_index(drop=True)

print(df_result)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(15, 6))
x = np.arange(len(df_result))
width = 0.35

# Bar plot
plt.bar(x - width/2, df_result['donor_fraction'], width, label='Healthy')
plt.bar(x + width/2, df_result['ipf_fraction'], width, label='MASLD')

# X축 설정
plt.xticks(x, df_result['cluster'], rotation=30, ha='right')
plt.ylabel('Fraction')
plt.title('T cell cluster proportion by condition')
plt.legend()

# p-value 별로 * 표시
for i, pval in enumerate(df_result['p_value']):
    if pval < 0.001:
        stars = '***'
    elif pval < 0.01:
        stars = '**'
    elif pval < 0.05:
        stars = '*'
    else:
        stars = ''

    if stars:
        max_height = max(df_result['donor_fraction'][i], df_result['ipf_fraction'][i])
        plt.text(x[i], 0.348, stars, ha='center', va='bottom', fontsize=12, color='black')

plt.tight_layout()
#plt.savefig("/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/05_Figure/Fig2B_proportionbargraph.pdf", bbox_inches='tight', dpi=300)

In [ ]:
adata.obs['celltype'] = adata.obs['leiden_recluster_r0.8']
adata.obs['celltype'] = adata.obs['celltype'].replace({
    '0': 'Classical_monocyte',
    '10': 'Classical_monocyte',
    '14': 'Classical_monocyte',
    '1': 'CD4_NaiveT',
    '4': 'CD4_NaiveT',
    '5': 'CD4_NaiveT',
    '2': 'CD8_EM',
    '8': 'CD8_EM',
    '3': 'NK',
    '6': 'B_cell',
    '7': 'CD8_NaiveT',
    '9': 'Inflammatory_Monocyte_1',
    '11': 'Non_classical_monocyte',
    '12': 'Treg',
    '13': 'CD8_EM',
    '15': 'CD8_EM',
    '15': 'CD8_EM',
    '16': 'cDC2',
    '17': 'Platelet',
    '18': 'pDC',
    '19': 'Erythrocyte',
    '20': 'Inflammatory_Monocyte_2',
    '21': 'Plasmablast',
    '22': 'HSPC',

})

sc.pl.umap(adata, color=['celltype'], legend_loc="on data")

In [ ]:
adata

In [ ]:
sc.pl.umap(adata, color=["leiden_recluster_r0.8"], title="leiden")

In [ ]:
sc.pl.umap(adata, color=["cellannot_final"], title="Original annotation")

In [ ]:
sc.pl.umap(adata, color=['celltype'])

In [ ]:
adata.obs['celltype'].value_counts()

In [ ]:
celltypes = adata.obs['celltype'].cat.categories
cluster_labels = adata.obs['celltype']
batch_labels = adata.obs['disease']

crosstab = pd.crosstab(batch_labels, cluster_labels, normalize='index')
crosstab = crosstab.reindex(['Normal', 'MASLD'])

crosstab.plot(
    kind='bar',
    stacked=True,
    figsize=(5, 10),
    #color=[palette_dict[col] for col in crosstab.columns]
)

plt.title('Major cell lineages in MASLD vs Normal')
plt.ylabel('Proportion')
plt.xlabel('state')
plt.legend(title='MASLD vs Normal', bbox_to_anchor=(1.05, 1), loc='upper left')
#plt.savefig(
#    "/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/05_Figure/Fig2B_stackedbargraph.pdf",
#    bbox_inches='tight',
#    dpi=300
#)

In [ ]:
# file: check_monocyte_signature_in_deg_pbmc.py

import scanpy as sc
import pandas as pd
import numpy as np

signature_genes = [
    "PPIF","ANPEP","ITGAX","FLNA","ITGA5","PNP","CD300E","PHLDA1","ATP13A3","SLC11A1",
    "RNASEK","SLC43A2","EZR","IFI30","SLC25A37","EEF1G","AHNAK","GPCPD1","BACH1","TOM1",
    "OLR1","PLEC","MT-ATP8","NME2","ATP2B1","NBPF26","ATP6V0C","LIMS1","GABARAPL1","RGCC",
    "MYADM","SPAG9","METRNL","MAP2K3","CDKN1A","DENND5A","SRRM2","FMNL1","SERPINB9","DUSP6",
    "HMGA1","EIF4A1","UPP1","ZMIZ1","TREM1","MYO9B","C5AR1","EMILIN2","NFKB1","ZYX",
    "CRIP1","CD44","TPM4","GABARAP","S100A10","EFHD2","LUCAT1","KDM6B","LILRB2","UBE2R2",
    "EMP3","FGR","SOD2","SPHK1","CSF3R","LHFPL2","WARS","FAM107B","TNFRSF1B","JUND",
    "TOMM6","APOBEC3A","TRAPPC5","ZBTB7A","PLK3","RARA","SQSTM1","GK","CDV3","ETV6",
    "YBX3","ODF3B","VCAN","IL6R","ATP1B3","TCIRG1","NCOR2","TIMP1","RNF213","DYNC1H1",
    "DSE","STK40","MAP3K8","TNFAIP2","EMD","LRRFIP1","PIM3","MAP4K4","MYO1G","NCF2",
    "MT-ND4L","ETF1","NINJ1","MAFB","JARID2","PPP1R15A","RAB24","SYAP1","FCN1","TAF10",
    "IER3","DMXL2","SKIL","DDX21","ARHGAP26","MAP3K2","CRTAP","AHR","GPR183","ELK3",
    "ATF5","BTG1","IL1RN","SH3BP5","ZNF385A","HIF1A","AQP9","BCL3","UBXN11","ACSL1",
    "CAST","CDC37","TGIF1","ANXA5","TNIP1","ANXA2","JMJD1C","VEGFA","NFKB2","TCF25",
    "GLIPR2","FPR1","SFPQ","MIR22HG","MAPK6","ASPH","LAIR1","PLEKHO2","EREG","DIAPH1",
    "LITAF","STK10","SON","KLF10","LPCAT1","UBE2V1","PRKAG2","MAN2B1","POLD4","CSGALNACT2",
    "JOSD1","CAPN2","DDX39B","PNPLA8","CSNK1G2","LFNG","OGDH","C17orf49","SERTAD1","KYNU",
    "ACADVL","ARAP1","GNA13","MYH9","HAVCR2","GNAI2","NBPF14","RAPGEF1","ZEB2","TET2",
    "MALT1","S100A8","GNA15","ALYREF","SMAD7","FERMT3","PSMA6","C15orf48","ARID5A","S100A9",
    "KCTD20","ATP1A1","TLE3","TGFB1","SLCO3A1","PITPNA","RNF145","RILPL2","LACTB","IRAK1",
    "CHMP4B","AGO2","ZBTB43","ADGRE5","YWHAZ","MBP","NUMB","ZFYVE16","SLC16A3","BCL2A1",
    "PKM","ADSS","AMPD2","IRF7","PLEKHM2","SLC7A5","GRK6","HM13","GSTO1","PTPRE",
    "PNPLA2","TPD52L2","LGALS3","PLXDC2","FOSL2","NDRG1","RALGDS","IRF2BP2","MARCKSL1","ARF6",
    "DOK3","CXCL8","IRAK3","GRB2","PHACTR1","ARRB2","RBM39","EMP1","LRP1","CREM",
    "ILF3","DNM2","TES","IL4R","CKAP4","MAFG","RC3H1","STAT6","SGK1","DNTTIP2",
    "UBALD2","SULF2","NRGN","ISG20","RAP1B","TRIM25","MAP2K1","MAP7D1","TUBB6","ATP6V0D1",
    "RNF19B","RANBP2","STK24","RCOR1","MAP2K2","ANKHD1","SHKBP1","ANXA1","MXD1","TUBB4B",
    "ARHGDIA","TUBA1C","SNN","EHBP1L1","RELB","HEBP2","BEST1","CYTH1","PLP2","MGAT1",
    "INSIG1","MRPS24","NAPA","GRINA","ZNF267","FAM214B","PRRC2C","PID1","VPS37B","STX11",
    "ADAM8","SERPINA1","TLN1","SMCHD1","RNF168","CUX1","EIF4G2","QKI","CXCL2","DEK"
]

signature_genes = [g.upper() for g in signature_genes]

deg = deg_pbmc.copy()
deg["names_upper"] = deg["names"].astype(str).str.upper()

# 1) adata에 실제 존재하는 signature gene 수
genes_in_adata = set(adata.raw.var_names.str.upper()) if adata.raw is not None else set(adata.var_names.str.upper())
signature_in_adata = sorted(set(signature_genes) & genes_in_adata)

print(f"Signature genes total: {len(signature_genes)}")
print(f"Signature genes found in adata: {len(signature_in_adata)}")

# 2) deg_pbmc 전체와 overlap
overlap_all = sorted(set(signature_genes) & set(deg["names_upper"]))
print(f"Signature genes found in deg_pbmc: {len(overlap_all)}")
print(overlap_all[:50])

# 3) cluster별 overlap 개수
overlap_by_group = (
    deg.assign(is_sig=deg["names_upper"].isin(signature_genes))
       .groupby("group")["is_sig"]
       .sum()
       .sort_values(ascending=False)
)
print("\nOverlap counts by cluster:")
print(overlap_by_group)

# 4) cluster별 상위 n개 marker 안에서 overlap 보기
top_n = 100
top_deg = deg.groupby("group", group_keys=False).head(top_n).copy()

overlap_topn_summary = (
    top_deg.assign(is_sig=top_deg["names_upper"].isin(signature_genes))
           .groupby("group")["is_sig"]
           .agg(["sum", "count"])
           .rename(columns={"sum": "n_overlap", "count": f"top_{top_n}"})
)
overlap_topn_summary["overlap_fraction"] = overlap_topn_summary["n_overlap"] / overlap_topn_summary[f"top_{top_n}"]
overlap_topn_summary = overlap_topn_summary.sort_values(["n_overlap", "overlap_fraction"], ascending=False)

print(f"\nOverlap in top {top_n} markers by cluster:")
print(overlap_topn_summary)

# 5) 어떤 gene이 겹치는지 cluster별로 보기
overlap_genes_by_group = (
    top_deg[top_deg["names_upper"].isin(signature_genes)]
    .groupby("group")["names"]
    .apply(list)
)

print(f"\nOverlapping genes in top {top_n} markers by cluster:")
for grp, genes in overlap_genes_by_group.items():
    print(f"{grp}: {genes}")

# 6) hypergeometric test로 enrichment 계산
from scipy.stats import hypergeom
from statsmodels.stats.multitest import multipletests

M = len(set(deg["names_upper"]))          # universe
n = len(set(signature_in_adata))          # signature size in data

enrich_rows = []
for grp, df_grp in top_deg.groupby("group"):
    group_genes = set(df_grp["names_upper"])
    N = len(group_genes)
    k = len(group_genes & set(signature_in_adata))

    pval = hypergeom.sf(k - 1, M, n, N)

    enrich_rows.append({
        "group": grp,
        "universe_M": M,
        "signature_n": n,
        "topN_N": N,
        "overlap_k": k,
        "pval": pval,
        "overlap_genes": sorted(group_genes & set(signature_in_adata)),
    })

enrich_df = pd.DataFrame(enrich_rows)
enrich_df["pval_adj"] = multipletests(enrich_df["pval"], method="fdr_bh")[1]
enrich_df = enrich_df.sort_values(["overlap_k", "pval_adj"], ascending=[False, True])

print("\nEnrichment test by cluster:")
print(enrich_df[["group", "overlap_k", "topN_N", "signature_n", "pval", "pval_adj"]])

# 7) signature score를 cell별로 계산
score_genes = [g for g in signature_in_adata if g in genes_in_adata]

# 대소문자 정렬 문제 방지
if adata.raw is not None:
    gene_map = {g.upper(): g for g in adata.raw.var_names}
else:
    gene_map = {g.upper(): g for g in adata.var_names}

score_genes_mapped = [gene_map[g] for g in score_genes if g in gene_map]

sc.tl.score_genes(
    adata,
    gene_list=score_genes_mapped,
    score_name="monocyte_signature_score",
    use_raw=True
)

# cluster별 평균 score
score_summary = (
    adata.obs.groupby("leiden_recluster_r0.8")["monocyte_signature_score"]
    .agg(["mean", "median", "count"])
    .sort_values("mean", ascending=False)
)
print("\nSignature score by cluster:")
print(score_summary)

In [ ]:
sc.pl.violin(
    adata,
    keys="monocyte_signature_score",
    groupby="leiden_recluster_r0.8",
    rotation=90
)

sc.pl.umap(
    adata,
    color=["leiden_recluster_r0.8", "monocyte_signature_score"],
    cmap="viridis"
)

## CellTypist re-annotation

CellTypist에는 log-normalized expression이 필요합니다. `layers["log1p_norm"]`이 있으면 그 값을 쓰고, 없으면 `layers["counts"]`에서 CellTypist용 log-normalized matrix를 새로 만듭니다.


In [ ]:
# -----------------------------------------------------------------------------
# Prepare log-normalized matrix for CellTypist
# -----------------------------------------------------------------------------
def make_celltypist_adata(source: ad.AnnData) -> ad.AnnData:
    if "log1p_norm" in source.layers:
        x = _copy_matrix(source.layers["log1p_norm"])
        var = source.var.copy()
        source_msg = "layers['log1p_norm']"
    elif "counts" in source.layers:
        ct_tmp = ad.AnnData(
            X=_copy_matrix(source.layers["counts"]),
            obs=source.obs.copy(),
            var=source.var.copy(),
        )
        sc.pp.normalize_total(ct_tmp, target_sum=1e4)
        sc.pp.log1p(ct_tmp)
        x = _copy_matrix(ct_tmp.X)
        var = ct_tmp.var.copy()
        source_msg = "layers['counts'] normalized for CellTypist"
    elif source.raw is not None:
        ct_tmp = source.raw.to_adata().copy()
        ct_tmp.obs = source.obs.copy()
        sc.pp.normalize_total(ct_tmp, target_sum=1e4)
        sc.pp.log1p(ct_tmp)
        x = _copy_matrix(ct_tmp.X)
        var = ct_tmp.var.copy()
        source_msg = "adata.raw normalized for CellTypist"
    else:
        x = _copy_matrix(source.X)
        var = source.var.copy()
        source_msg = "adata.X as-is; verify this is log-normalized"

    ct = ad.AnnData(X=x, obs=source.obs.copy(), var=var)
    ct.obsm["X_umap"] = source.obsm[UMAP_KEY].copy()
    print(f"CellTypist input: {source_msg}")
    return ct


adata_ct = make_celltypist_adata(adata_re)
print(adata_ct)


In [ ]:
# -----------------------------------------------------------------------------
# Run CellTypist
# -----------------------------------------------------------------------------
if RUN_CELLTYPIST:
    import celltypist
    from celltypist import models

    if DOWNLOAD_CELLTYPIST_MODEL:
        models.download_models(model=CELLTYPIST_MODEL, force_update=False)

    predictions = celltypist.annotate(
        adata_ct,
        model=CELLTYPIST_MODEL,
        majority_voting=True,
        over_clustering=CLUSTER_KEY,
    )

    celltypist_labels = predictions.predicted_labels.copy()
    display(celltypist_labels.head())

    adata_re.obs["celltypist_predicted_recluster"] = celltypist_labels["predicted_labels"].astype("category")
    if "majority_voting" in celltypist_labels.columns:
        adata_re.obs["celltypist_majority_recluster"] = celltypist_labels["majority_voting"].astype("category")
    else:
        adata_re.obs["celltypist_majority_recluster"] = adata_re.obs["celltypist_predicted_recluster"]

    sc.pl.embedding(
        adata_re,
        basis="umap_recluster",
        color=[CLUSTER_KEY, "celltypist_predicted_recluster", "celltypist_majority_recluster"],
        wspace=0.45,
        ncols=3,
    )
else:
    print("RUN_CELLTYPIST=False. Skipping CellTypist.")


In [ ]:
# -----------------------------------------------------------------------------
# Cluster-level annotation summary
# -----------------------------------------------------------------------------
if "celltypist_majority_recluster" in adata_re.obs.columns:
    cluster_annotation_table = pd.crosstab(
        adata_re.obs[CLUSTER_KEY],
        adata_re.obs["celltypist_majority_recluster"],
        normalize="index",
    ).round(3)
    display(cluster_annotation_table)

    top_celltypist_by_cluster = cluster_annotation_table.idxmax(axis=1).to_dict()
    top_celltypist_by_cluster = {str(k): str(v) for k, v in top_celltypist_by_cluster.items()}
else:
    cluster_annotation_table = pd.DataFrame()
    top_celltypist_by_cluster = {}

print("Top CellTypist label by cluster")
display(pd.Series(top_celltypist_by_cluster, name="celltypist_top_label").to_frame())

if ORIGINAL_LABEL_COLS:
    for original_col in ORIGINAL_LABEL_COLS:
        comparison = pd.crosstab(
            adata_re.obs[CLUSTER_KEY],
            adata_re.obs[original_col],
            normalize="index",
        ).round(3)
        print(f"\nRecluster vs original {original_col}")
        display(comparison)


In [ ]:
# -----------------------------------------------------------------------------
# Final annotation column
# -----------------------------------------------------------------------------
cluster_to_label = top_celltypist_by_cluster.copy()
cluster_to_label.update({str(k): str(v) for k, v in MANUAL_CLUSTER_LABELS.items()})

adata_re.obs["cellannot_recluster"] = (
    adata_re.obs[CLUSTER_KEY]
    .astype(str)
    .map(cluster_to_label)
    .fillna("Unknown")
    .astype("category")
)

print("Final cluster_to_label mapping")
display(pd.Series(cluster_to_label, name="cellannot_recluster").to_frame())

sc.pl.embedding(
    adata_re,
    basis="umap_recluster",
    color=[CLUSTER_KEY, "cellannot_recluster"],
    wspace=0.4,
)


In [ ]:
# -----------------------------------------------------------------------------
# Save reclustered object and summary tables
# -----------------------------------------------------------------------------
out_h5ad = OUT_DIR / f"{SAMPLE_NAME}_reclustered_celltypist.h5ad"
out_markers = OUT_DIR / f"{SAMPLE_NAME}_reclustered_rank_genes.csv"
out_cluster_counts = OUT_DIR / f"{SAMPLE_NAME}_reclustered_cluster_counts.csv"
out_celltypist_crosstab = OUT_DIR / f"{SAMPLE_NAME}_reclustered_celltypist_crosstab.csv"
out_obs = OUT_DIR / f"{SAMPLE_NAME}_reclustered_obs_annotations.csv"

adata_re.write_h5ad(out_h5ad)
rank_df.to_csv(out_markers, index=False)
adata_re.obs[CLUSTER_KEY].value_counts().sort_index().to_csv(out_cluster_counts, header=["n_cells"])

if not cluster_annotation_table.empty:
    cluster_annotation_table.to_csv(out_celltypist_crosstab)

obs_cols = [CLUSTER_KEY, "cellannot_recluster"]
obs_cols += [col for col in ["celltypist_predicted_recluster", "celltypist_majority_recluster"] if col in adata_re.obs.columns]
obs_cols += [col for col in ORIGINAL_LABEL_COLS if col in adata_re.obs.columns]
adata_re.obs[obs_cols].to_csv(out_obs)

print("Saved files:")
for path in [out_h5ad, out_markers, out_cluster_counts, out_celltypist_crosstab, out_obs]:
    if path.exists():
        print(f"- {path}")
